# AI-QMS — Wait-Time Prediction (Phase 5)

Random Forest (primary, tuned max_depth=6, min_samples_leaf=20) vs. a linear regression baseline,
trained on the Phase 2 clean data (5,768 rows, 3 sources: hospital / queue log / survey).

- **Models**: ackend/app/ml/models/wait_time_model.joblib (promoted Random Forest), wait_time_baseline_linear.joblib (kept for the Phase 9 comparison)
- **Peak/off-peak**: pandas groupby on hour-of-day × day-of-week → peak_table.csv
- XGBoost/LightGBM intentionally skipped: 5.8k rows too small to justify (architecture.md §2)

Run headlessly with: python -m pipeline.train then python -m pipeline.peaks (from ml-notebooks/).

In [1]:
from pipeline.train import load_clean, feature_columns, evaluate, RF_CONFIG
import pipeline.train as train

df = load_clean()
features, service_cols = feature_columns(df)
df.describe()[['wait_time_min', 'queue_length_at_arrival', 'hour_of_day']]

,wait_time_min,queue_length_at_arrival,hour_of_day
count,5768.000000,5768.000000,5768.000000
mean,54.488124,3.510142,11.602982
std,45.926241,10.524305,3.337624
min,0.000000,0.000000,0.000000
25%,22.000000,0.000000,9.000000
50%,42.000000,0.000000,12.000000
75%,75.000000,0.000000,14.000000
max,240.000000,50.000000,23.000000


In [2]:
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_score

X = df[features].to_numpy(dtype=float)
y = df['wait_time_min'].to_numpy(dtype=float)

sorted_idx = df.sort_values('arrival_ts').index
split = int(train.TRAIN_FRACTION * len(df))
Xt, Xe, yt, ye = X[sorted_idx[:split]], X[sorted_idx[split:]], y[sorted_idx[:split]], y[sorted_idx[split:]]

rf = RandomForestRegressor(**RF_CONFIG, n_jobs=-1).fit(Xt, yt)
lr = LinearRegression().fit(Xt, yt)
rf_results = evaluate(rf, Xe, ye)
lr_results = evaluate(lr, Xe, ye)
rf_results, lr_results

({'mae_min': 30.84, 'rmse_min': 43.2, 'r2': 0.1285},
 {'mae_min': 31.38, 'rmse_min': 43.3, 'r2': 0.1244})

In [3]:
cv = KFold(n_splits=5, shuffle=True, random_state=train.RANDOM_STATE)
rf_cv = -cross_val_score(rf, X, y, cv=cv, scoring='neg_root_mean_squared_error').mean()
lr_cv = -cross_val_score(lr, X, y, cv=cv, scoring='neg_root_mean_squared_error').mean()
print(f'5-fold CV RMSE (min): RF {rf_cv:.2f}  LR {lr_cv:.2f}')

5-fold CV RMSE (min): RF 41.90  LR 41.71


In [4]:
from pipeline.peaks import build_peak_table

table, overall_mean = build_peak_table(df)
print(f'overall mean wait: {overall_mean:.2f} min')
table[table['is_peak']].sort_values('avg_wait_min', ascending=False).head(10)

overall mean wait: 54.49 min


,hour_of_day,day_of_week,avg_wait_min,count,is_peak
25,7,5,74.631579,19,True
69,14,0,70.936782,87,True
26,7,6,69.714286,21,True
76,15,0,69.448718,78,True
73,14,4,66.879121,91,True
21,7,1,66.592593,27,True
45,10,4,66.053333,75,True
28,8,1,66.036842,95,True
67,13,5,65.797753,89,True
70,14,1,65.672619,84,True


In [5]:
peak_hours = table[table['is_peak']]['hour_of_day'].sort_values().unique()
f'peak hours flagged: {peak_hours.tolist()}  |  weekend peak hours: {table[table['is_peak'] & (table['day_of_week'] >= 5)]['hour_of_day'].sort_values().unique().tolist()}' 

'peak hours flagged: [7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]  |  weekend peak hours: [7, 8, 9, 10, 11, 12, 13, 14, 15, 16]'

## Outcome

RF and LR are statistically indistinguishable on this noisy self-reported dataset
(test RMSE 43.2 vs 43.3 min, R² ≈ 0.13 vs 0.12). Per architecture.md §2 the tuned
Random Forest is promoted as wait_time_model.joblib; the linear baseline artifact
is kept for the Step 6 improved-vs-traditional comparison. Both are served from
ackend/app/ml/ by predictor.py (with peak-table fallback) and consumed by
optimizer.py for counter/staff allocation suggestions.